You are working with two complementary datasets describing a large-scale biotechnological production process.

The first dataset (“4000 series operating data”) contains high-frequency sensor measurements collected every 5–15 minutes over several hundred hours for 22 individual batches. It includes 17 process variables such as liquid inflows, gas inflows, pH, off-gas measurements, pressure, and oxygen levels. In total, the dataset contains approximately 1.4 million time-stamped observations. Some values are missing due to sensor limitations or non-operational readings, but all batches are considered representative of normal process operation.

The second dataset contains laboratory measurements recorded approximately every 4 hours for 21 batches. This dataset includes the product concentration values over time, which are used to calculate a yield parameter for each batch. By combining the time-series sensor data with the laboratory product data, it is possible to construct a data-driven model to identify which process variables most strongly influence product yield and to predict the yield of a previously unseen batch.

Import data

In [17]:
from applied.data_processing import (
    load_operating_data,
    load_product_data,
    build_features_and_target,
)

In [18]:
from pathlib import Path

# Project root = one level above notebooks/
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

operating_csv = DATA_DIR / "4000 series operating data.csv"
product_xlsx = DATA_DIR / "4000 series product data.xlsx"

In [19]:
op_df = load_operating_data(operating_csv)
op_df.head()

,Date and time,Batch,LIQUID,LIQUID.1,LIQUID.2,LIQUID.3,LIQUID.4,LIQUID.5,pH,GAS,GAS.1,GAS.2,GAS.3,OFFGAS,OFFGAS.1,PRESSURE,PRESSURE.1,OXYGEN
2,2019-04-02 08:46:00,4030,1049.57,25.91,14.44,NaN,297.73,14980.0,5.76,NaN,NaN,NaN,55.14,1.87,16.94,1.79,5.15,6.78
3,2019-04-02 09:01:00,4030,1049.09,25.62,13.54,NaN,357.44,15000.0,5.79,NaN,NaN,NaN,56.88,1.89,20.52,1.80,5.16,8.39
4,2019-04-02 09:16:00,4030,1049.61,25.44,13.59,NaN,356.83,15010.0,5.80,NaN,NaN,NaN,56.03,1.94,23.77,1.80,5.15,8.07
5,2019-04-02 09:31:00,4030,1047.57,25.59,13.97,NaN,356.77,15000.0,5.79,NaN,NaN,NaN,53.91,2.00,27.01,1.80,5.15,7.23
6,2019-04-02 09:46:00,4030,1048.16,25.49,13.43,NaN,357.21,15010.0,5.78,NaN,NaN,NaN,53.97,2.05,30.15,1.80,5.13,7.16


The operating dataset contains 83,204 time-series observations across 18 columns. Each row corresponds to a timestamped measurement for a specific batch. The Batch column is an integer identifier, Date and time is stored as a datetime variable, and all process variables (LIQUID, GAS, OFFGAS, PRESSURE, pH, and OXYGEN) are continuous numerical features stored as float64. Some missing values are present due to sensor downtime or non-operational readings.

To reduce dimensionality and improve interpretability, related sensor streams were aggregated. The six liquid inflow variables were summed into TOTAL_LIQUID_INFLOW, the four gas inflow variables into TOTAL_GAS_INFLOW, the two off-gas sensors were averaged into TOTAL_OFFGAS, and the two pressure sensors were averaged into MEAN_PRESSURE. This transformation reduces redundancy from multiple sensors measuring similar physical phenomena, mitigates noise between parallel sensors, and simplifies the feature space while preserving the key process information relevant for yield prediction.

Feature Aggregation and Dimensionality Reduction

The operating dataset contains multiple sensor measurements representing similar physical quantities recorded from parallel inflow streams and duplicate sensors. To improve interpretability and reduce redundancy, related variables were aggregated into composite features.

The six liquid inflow variables (LIQUID, LIQUID.1, LIQUID.2, LIQUID.3, LIQUID.4, LIQUID.5) represent separate feed streams entering the same bioreactor. Since the overall system behaviour depends on the total liquid entering the reactor, these variables were summed to create a single feature, TOTAL_LIQUID_INFLOW. Similarly, the four gas inflow variables (GAS, GAS.1, GAS.2, GAS.3) were summed to form TOTAL_GAS_INFLOW, representing the total gas supply to the system.

In contrast, the OFFGAS and PRESSURE variables represent duplicate sensors measuring the same physical process conditions. These were averaged to create TOTAL_OFFGAS and MEAN_PRESSURE, respectively. Averaging improves robustness by reducing measurement noise and mitigating small calibration differences between sensors, while preserving the true underlying process state.

Following aggregation, the original individual sensor columns were removed to reduce dimensionality and multicollinearity. This simplification improves model stability, reduces the risk of overfitting, and ensures that the features more directly reflect the underlying physical processes influencing product yield.

In [20]:
# --- Create totals ---

op_df["TOTAL_LIQUID_INFLOW"] = op_df[
    ["LIQUID", "LIQUID.1", "LIQUID.2",
     "LIQUID.3", "LIQUID.4", "LIQUID.5"]
].sum(axis=1)

op_df["TOTAL_GAS_INFLOW"] = op_df[
    ["GAS", "GAS.1", "GAS.2", "GAS.3"]
].sum(axis=1)

op_df["TOTAL_OFFGAS"] = op_df[
    ["OFFGAS", "OFFGAS.1"]
].mean(axis=1)   # mean is better than sum for sensors

op_df["MEAN_PRESSURE"] = op_df[
    ["PRESSURE", "PRESSURE.1"]
].mean(axis=1)


# --- Drop original columns ---

op_df = op_df.drop(columns=[
    "LIQUID", "LIQUID.1", "LIQUID.2",
    "LIQUID.3", "LIQUID.4", "LIQUID.5",
    "GAS", "GAS.1", "GAS.2", "GAS.3",
    "OFFGAS", "OFFGAS.1",
    "PRESSURE", "PRESSURE.1"
])

In [27]:
op_df.columns

Index(['Date and time', 'Batch', 'pH', 'OXYGEN', 'TOTAL_LIQUID_INFLOW',
       'TOTAL_GAS_INFLOW', 'TOTAL_OFFGAS', 'MEAN_PRESSURE'],
      dtype='object')

In [24]:
op_groups = op_df.groupby("Batch", sort=False)

for i, (batch_id, batch_df) in enumerate(op_groups):
    print(f"\nBatch: {batch_id}")
    print(batch_df.head())
    
    if i == 1:   # stop after 3 batches
        break


Batch: 4030
        Date and time  Batch    pH  OXYGEN  TOTAL_LIQUID_INFLOW  \
2 2019-04-02 08:46:00   4030  5.76    6.78             16367.65   
3 2019-04-02 09:01:00   4030  5.79    8.39             16445.69   
4 2019-04-02 09:16:00   4030  5.80    8.07             16455.47   
5 2019-04-02 09:31:00   4030  5.79    7.23             16443.90   
6 2019-04-02 09:46:00   4030  5.78    7.16             16454.29   

   TOTAL_GAS_INFLOW  TOTAL_OFFGAS  MEAN_PRESSURE  
2             55.14         9.405          3.470  
3             56.88        11.205          3.480  
4             56.03        12.855          3.475  
5             53.91        14.505          3.475  
6             53.97        16.100          3.465  

Batch: 4032
           Date and time  Batch    pH  OXYGEN  TOTAL_LIQUID_INFLOW  \
3073 2019-01-04 14:16:00   4032  5.84   10.80             15958.48   
3074 2019-01-04 14:31:00   4032  5.86   33.03             14358.70   
3075 2019-01-04 14:46:00   4032  5.74   29.76          

calculate tagret metric

In [31]:
prod_df = load_product_data(product_xlsx)
prod_groups = prod_df.groupby("Batch", sort=False)

In [32]:
prod_df

for i, (batch_id, batch_df) in enumerate(prod_groups):
    print(f"\nBatch: {batch_id}")
    print(batch_df.head())
    
    if i == 1:   # stop after 3 batches
        break


Batch: 4030
        Date and time  Batch  Product
2 2019-02-04 00:00:00   4030      5.9
3 2019-02-04 02:00:00   4030      8.2
4 2019-02-04 04:00:00   4030      9.7
5 2019-02-04 06:00:00   4030     14.3
6 2019-02-04 08:00:00   4030     16.4

Batch: 4032
          Date and time  Batch  Product
194 2019-04-01 14:00:00   4032     18.6
195 2019-04-01 15:00:00   4032     18.8
196 2019-04-01 16:30:00   4032     20.0
197 2019-04-01 20:00:00   4032     18.2
198 2019-04-02 00:00:00   4032     20.8


1️⃣ The Simplest Approach (Baseline Model)
Use statistical summaries per batch:
batch_features = op_df.groupby("Batch").agg([
    "mean",
    "std",
    "max",
    "min"
])

Why this works:

Mean → overall operating level

Std → process stability

Max/Min → extreme operating conditions

This is a strong baseline and often surprisingly effective.

2️⃣ Better: Add Dynamic Information

Bioprocesses are time-dependent. Yield is often influenced by:

Early phase conditions

Growth phase transitions

Process instability

Oxygen limitation events

So instead of only using global mean, you can engineer:

✅ Phase-based features

Split each batch into phases:

def early_phase(x):
    return x.iloc[:int(len(x)*0.2)].mean()

def late_phase(x):
    return x.iloc[int(len(x)*0.8):].mean()


This captures:

Start-up behaviour

End-of-run stress conditions

✅ Trend features (very powerful)

Compute slope over time:

from scipy.stats import linregress

def slope_feature(x):
    return linregress(range(len(x)), x).slope


Why this matters:

Rising oxygen demand?

Increasing gas requirement?

Falling pH trend?

Trend often predicts yield better than averages.

✅ Variability features

Biological systems hate instability.

Compute:

Rolling standard deviation

Coefficient of variation

Number of threshold breaches

Example:

cv = x.std() / x.mean()

3️⃣ What I Recommend for Your Case Study

Since this is a data-driven industrial setting, I would build:

Core Features:

Mean of each variable

Std of each variable

Add:

Slope of pH

Slope of OXYGEN

Std of TOTAL_GAS_INFLOW

Early-phase mean oxygen

That gives:

Process level

Stability

Dynamic trend

Early biological conditions

That is strong and still explainable.

4️⃣ Why This Is Important for Yield Prediction

Yield is not caused by:

a single value at a single time

It is caused by:

how the process behaves over time.

If you only use mean values, you may miss:

Oxygen starvation periods

Gas spikes

Pressure instability

Drift in pH

5️⃣ If You Want a Clean Report Sentence

You could write:

Since product yield is determined by the dynamic behaviour of the process rather than single-point measurements, batch-level statistical and trend-based features were extracted from the time-series data. These include measures of central tendency, variability, and temporal evolution, enabling the construction of a robust data-driven yield prediction model.

In [6]:
groups = op_df.groupby("Batch", sort=False)

In [7]:
batch_ids = groups.groups.keys()
batch_ids

dict_keys([4030, 4032, 4033, 4034, 4035, 4036, 4037, 4038, 4039, 4040, 4041, 4042, 4043, 4044, 4045, 4046, 4047, 4048, 4050, 4051, 4052, 4053])

In [8]:
batch_df = groups.get_group(4030)

batch_df.head()

,Date and time,Batch,pH,OXYGEN,TOTAL_LIQUID_INFLOW,TOTAL_GAS_INFLOW,TOTAL_OFFGAS,MEAN_PRESSURE
2,2019-04-02 08:46:00,4030,5.76,6.78,16367.65,55.14,9.405,3.470
3,2019-04-02 09:01:00,4030,5.79,8.39,16445.69,56.88,11.205,3.480
4,2019-04-02 09:16:00,4030,5.80,8.07,16455.47,56.03,12.855,3.475
5,2019-04-02 09:31:00,4030,5.79,7.23,16443.90,53.91,14.505,3.475
6,2019-04-02 09:46:00,4030,5.78,7.16,16454.29,53.97,16.100,3.465


In [9]:
prod_df = load_product_data(product_xlsx)
prod_df.head()

,Date and time,Batch,Product
2,2019-02-04 00:00:00,4030,5.9
3,2019-02-04 02:00:00,4030,8.2
4,2019-02-04 04:00:00,4030,9.7
5,2019-02-04 06:00:00,4030,14.3
6,2019-02-04 08:00:00,4030,16.4


### Aggregating Parallel Sensor Channels

In the operating data, several process variables (e.g. liquid inflow,
gas inflow, pressure) are measured using multiple parallel sensor
channels. These channels correspond to different physical streams or
sensors measuring the same underlying quantity, rather than distinct
process variables.

For example, the liquid inflow is recorded across six parallel channels
(`LIQUID_1`–`LIQUID_6`). Since all channels represent contributions to the
same physical inflow, it is appropriate to aggregate them into a single
total liquid inflow variable.

The aggregation is performed **at each measurement timestamp**. The
total liquid inflow at time $t$ is defined as

$$
\text{LIQUID}_{\text{total}}(t)
=
\sum_{i=1}^{6} \text{LIQUID}_i(t)
$$

If one or more channels are missing at a given time, the total inflow is
computed as the sum of the available channels only. Missing channel
values are not treated as zero, which avoids artificially inflating or
deflating the estimated inflow.

This aggregation is physically meaningful, reduces redundancy and
multicollinearity in the feature space, and aligns with the coursework
specification, which defines the product rate using the *total* liquid
inflow rather than individual streams.

A similar aggregation strategy is applied to other parallel sensor
groups (e.g. gas inflow and pressure), using summation or averaging as
appropriate based on the physical interpretation of each variable.


In [10]:
# --- Create totals ---

op_df["TOTAL_LIQUID_INFLOW"] = op_df[
    ["LIQUID", "LIQUID.1", "LIQUID.2",
     "LIQUID.3", "LIQUID.4", "LIQUID.5"]
].sum(axis=1)

op_df["TOTAL_GAS_INFLOW"] = op_df[
    ["GAS", "GAS.1", "GAS.2", "GAS.3"]
].sum(axis=1)

op_df["TOTAL_OFFGAS"] = op_df[
    ["OFFGAS", "OFFGAS.1"]
].mean(axis=1)   # mean is better than sum for sensors

op_df["MEAN_PRESSURE"] = op_df[
    ["PRESSURE", "PRESSURE.1"]
].mean(axis=1)


# --- Drop original columns ---

op_df = op_df.drop(columns=[
    "LIQUID", "LIQUID.1", "LIQUID.2",
    "LIQUID.3", "LIQUID.4", "LIQUID.5",
    "GAS", "GAS.1", "GAS.2", "GAS.3",
    "OFFGAS", "OFFGAS.1",
    "PRESSURE", "PRESSURE.1"
])

KeyError: "None of [Index(['LIQUID', 'LIQUID.1', 'LIQUID.2', 'LIQUID.3', 'LIQUID.4', 'LIQUID.5'], dtype='object')] are in the [columns]"

In [10]:
op_df

,Date and time,Batch,pH,OXYGEN,TOTAL_LIQUID_INFLOW,TOTAL_GAS_INFLOW,TOTAL_OFFGAS,MEAN_PRESSURE
2,2019-04-02 08:46:00,4030,5.76,6.78,16367.65,55.14,9.405,3.470
3,2019-04-02 09:01:00,4030,5.79,8.39,16445.69,56.88,11.205,3.480
4,2019-04-02 09:16:00,4030,5.80,8.07,16455.47,56.03,12.855,3.475
5,2019-04-02 09:31:00,4030,5.79,7.23,16443.90,53.91,14.505,3.475
6,2019-04-02 09:46:00,4030,5.78,7.16,16454.29,53.97,16.100,3.465
...,...,...,...,...,...,...,...,...
83201,NaT,4053,5.83,113.78,13799.79,11741.07,2.590,3.555
83202,NaT,4053,5.72,114.99,21098.71,11347.51,2.555,3.540
83203,NaT,4053,5.62,115.00,21840.41,10896.03,2.570,3.530
83204,NaT,4053,5.53,115.00,21838.40,10562.03,2.555,3.520
